In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
def get_input_id(real_entry_length, max_seq_length, padding_id=0):
    """
    随机生成token id
    """
    input_id = torch.randint(1, 100, (1, max_seq_length)).long()
    input_id[:, real_entry_length:max_seq_length] = padding_id  # Pad the rest with zeros
    
    return input_id

def get_position_id(input_id, padding_id=0):
    """
    
    """
    input_mask = input_id.ne(padding_id).int()
    position_id = torch.cumsum(input_mask, dim=-1) * input_mask
    return position_id

    
for i in range(10):
    real_entry_length = np.random.randint(1, 15)
    input_id = get_input_id(real_entry_length,15)
    position_id = get_position_id(input_id)
    print(f"real_entry_length:{real_entry_length}, input_id:", input_id)
    print(f"position_id:", position_id)

real_entry_length:13, input_id: tensor([[44, 44, 83, 41, 74, 65, 42, 79, 89, 90, 37, 39, 73,  0,  0]])
position_id: tensor([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13,  0,  0]])
real_entry_length:6, input_id: tensor([[45, 81, 98, 11, 79, 27,  0,  0,  0,  0,  0,  0,  0,  0,  0]])
position_id: tensor([[1, 2, 3, 4, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
real_entry_length:8, input_id: tensor([[ 2, 33, 11, 65, 84, 98, 45, 23,  0,  0,  0,  0,  0,  0,  0]])
position_id: tensor([[1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 0, 0, 0, 0]])
real_entry_length:9, input_id: tensor([[58, 35, 80, 12, 19, 96, 80, 61, 12,  0,  0,  0,  0,  0,  0]])
position_id: tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 0, 0, 0, 0, 0]])
real_entry_length:6, input_id: tensor([[50, 60, 81, 59, 42,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0]])
position_id: tensor([[1, 2, 3, 4, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
real_entry_length:7, input_id: tensor([[16, 86, 21, 25, 24, 31, 47,  0,  0,  0,  0,  0,  0,  0,  0]])
position_id: tensor([[1, 2,

In [18]:
# ------------------------------- 有关词嵌入 ------------------------------
vocab_size = 532100
hidden_size = 768
pad_token_id = 0
word_embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)

# ------------------------------ 有关位置嵌入 (1D)------------------------------
max_position_embeddings = 512
hidden_size = 768
padding_idx = 0
position_embeddings = nn.Embedding(
        max_position_embeddings, hidden_size, padding_idx)
# ------------------------------ 有关位置嵌入 (2D)------------------------------
max_2d_position_embeddings = 1024
coordinate_size = hidden_size//6
shape_size = hidden_size//6

x_position_embeddings = nn.Embedding(max_2d_position_embeddings, coordinate_size)
y_position_embeddings = nn.Embedding(max_2d_position_embeddings, coordinate_size)
h_position_embeddings = nn.Embedding(max_2d_position_embeddings, shape_size)
w_position_embeddings = nn.Embedding(max_2d_position_embeddings, shape_size)

In [ ]:
real_entry_length = np.random.randint(1, 15)
max_seq_length = 15
# token_id
input_id = get_input_id(real_entry_length,max_seq_length=max_seq_length)
# position_id
position_id = get_position_id(input_id)
# bbox 假设所有的token都会拥有一个bbox （在实际工程中可能所有的token共享同一个bbox）
# torch.randint(1, 100, (1, max_seq_length)).long()
bbox = torch.randint(1,100,(1, max_seq_length, 4))

left_position_embeddings = x_position_embeddings(bbox[:,:,0])
upper_position_embeddings = y_position_embeddings(bbox[:,:,1])
right_position_embeddings = x_position_embeddings(bbox[:, :, 2]) 
lower_position_embeddings = y_position_embeddings(bbox[:, :, 3])
h_position_embeddings = h_position_embeddings(torch.clip(bbox[:, :, 3] - bbox[:, :, 1], 0, 1023)) 
w_position_embeddings = w_position_embeddings(torch.clip(bbox[:, :, 2] - bbox[:, :, 0], 0, 1023))

# 拼接 .shape 为 (batch_size, max_seq_length, hidden_size//6 * 6)
bbox_embeddings = torch.cat([left_position_embeddings, upper_position_embeddings, right_position_embeddings, lower_position_embeddings, h_position_embeddings, w_position_embeddings], dim=-1)

out = word_embeddings(input_id) + position_embeddings(position_id) + bbox_embeddings   # 相加前后的维度必须相同，只是数值上发生变化
out.shape

torch.Size([1, 15, 768])

tensor([[63,  8, 26, 72, 63, 54,  7, 79, 94, 28, 82, 57,  6, 11,  0]])

tensor([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  0]])